# GraphST Benchmark for spatial mutliomcis data integration on simulated dataset

Notebook benchmarks spatial mutliomcis data integration using GraphST on simulated dataset.

## Loading

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import omicverse as ov
import anndata as ad
import pandas as pd
import scanpy as sc
import numpy as np

## GraphST pipeline

In [ ]:
# Set the directory for the datas                                                   ts and the output directory
data_dir = 'Original_Simulated_Data'
output_dir = 'Processed_Simulated_Data'
os.makedirs(output_dir, exist_ok=True)

# Loop through each dataset
for i in range(1, 6):
    print(f"Process {data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad data.")
    # Read the RNA dataset
    adata_rna = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad')
    adata_rna = adata_rna.raw.to_adata()
    adata_rna.obs['ground_truth'] = adata_rna.obs['cell_type']

    # Identify highly variable genes
    sc.pp.highly_variable_genes(adata_rna, n_top_genes=3000)
    adata_rna = adata_rna[:, adata_rna.var['highly_variable'] == True]

    # Set method parameters
    methods_kwargs = {}
    methods_kwargs['GraphST'] = {
        'device': 'cuda:0',
        'n_pcs': 30
    }

    # Perform spatial clustering using GraphST
    adata_rna = ov.space.clusters(adata_rna,
                                  methods=['GraphST'],
                                  methods_kwargs=methods_kwargs,
                                  lognorm=1e4)

    # Compute neighbors and perform clustering
    ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['GraphST_embedding'].shape[1],
                    use_rep='GraphST_embedding')
    ov.utils.cluster(adata_rna, use_rep='GraphST_embedding', method='leiden', resolution=0.15)

    # Plot spatial clustering results
    sc.pl.spatial(adata_rna, color=['ground_truth', 'leiden'], spot_size=0.12, wspace=0.4)

    # Save the processed dataset
    output_path = f'{output_dir}/Simulated_Dataset_{i}/graphst_rna.h5ad'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    adata_rna.write_h5ad(output_path, compression='gzip')

In [ ]:
!pip list